In [1]:
# !pip install transformers

In [1]:
import torch
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from transformers import YolosForObjectDetection, YolosImageProcessor
# from ultralytics import YOLOvXX
# from transformers import YolosFeatureExtractor


import torch.nn as nn
import torch.optim as optim

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/Users/imbilalbutt/virtual-env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
torch.__version__

'2.2.2'

In [3]:
import transformers
transformers.__version__

'4.49.0'

In [2]:
# 1. Load YOLOv5 model from Hugging Face
model_name = "hustvl/yolos-base" # "Ultralytics/YOLO11"
model = YolosForObjectDetection.from_pretrained(model_name)

In [5]:
model

YolosForObjectDetection(
  (vit): YolosModel(
    (embeddings): YolosEmbeddings(
      (patch_embeddings): YolosPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
      (interpolation): InterpolateInitialPositionEmbeddings()
    )
    (encoder): YolosEncoder(
      (layer): ModuleList(
        (0-11): 12 x YolosLayer(
          (attention): YolosSdpaAttention(
            (attention): YolosSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): YolosSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )


In [6]:
# YOLO builtin class labels (last layer neurons)
model.config.num_labels

91

In [7]:
model.class_labels_classifier

YolosMLPPredictionHead(
  (layers): ModuleList(
    (0-1): 2 x Linear(in_features=768, out_features=768, bias=True)
    (2): Linear(in_features=768, out_features=92, bias=True)
  )
)

In [8]:
model.config.hidden_size

768

In [9]:
model.config.hidden_size

768

In [3]:
# 2. Modify the output layer to have 5 classes (for CIFAR-10 subset)
# YOLO typically has (5 + num_classes) * num_anchors outputs per grid cell
# We'll modify the classifier head to output 5 classes
num_classes = 5
model.config.num_labels = num_classes

# Replace the classification head
model.class_labels_classifier = nn.Sequential(
    nn.Linear(model.config.hidden_size, model.config.hidden_size), # 768 x 768
    nn.ReLU(),
    nn.Linear(model.config.hidden_size, num_classes) # 768 x 5
)

model = model.to(device)


In [6]:
# 3. Prepare CIFAR-10 dataset (we'll use a subset with 5 classes)

# YOLOv5 expects images in RGB format with pixel values normalized to 0-1
transform = transforms.Compose([
    transforms.Resize((640, 640)),  # YOLOv5 default input size
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# We'll use only first 5 classes of CIFAR-10 for this example
class_subset = [0, 1, 2, 3, 4]  # airplane, automobile, bird, cat, deer


In [8]:
# Create training and test datasets
trainset = torchvision.datasets.CIFAR10(root='./data',
                                        train=True,
                                        download=False,
                                        transform=transform)

# trainset = [(img, label) for img, label in trainset if label in class_subset]


indices = [i for i, (_, label) in enumerate(trainset) if label in class_subset]

In [9]:
testset = torchvision.datasets.CIFAR10(root='./data',
                                       train=False,
                                       download=True,
                                       transform=transform)

# testset = [(img, label) for img, label in testset if label in class_subset]

test_indices = [i for i, (_, label) in enumerate(testset) if label in class_subset]

Files already downloaded and verified


In [10]:
from torch.utils.data import Subset

trainset = Subset(trainset, indices)

testset = Subset(testset, test_indices)

In [11]:
# Create data loaders
batch_size = 16

trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False)

In [12]:
# 4. Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# 5. Fine-tuning loop
num_epochs = 5

for epoch in range(num_epochs):
    running_loss = 0.0
    model.train()

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model(pixel_values=inputs)
        logits = outputs.logits

        # Compute loss (simplified for this example - in reality YOLO has complex loss)
        loss = criterion(logits, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()


    # model.train()
    # for imgs, targets in trainloader:
    #     imgs = imgs.to(device)
    #     targets = targets.to(device)
    #
    #     loss, loss_items = model(imgs, targets)  # YOLOv5 returns loss directly
    #
    #     loss.backward()
    #     optimizer.step()
    #     optimizer.zero_grad()

        if i % 100 == 99:  # Print every 100 mini-batches
            print(f"[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 100:.3f}")
            running_loss = 0.0

print("Finished Fine-Tuning")



In [ ]:
# 6. Evaluation
correct = 0
total = 0
model.eval()

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model(pixel_values=images)
        _, predicted = torch.max(outputs.logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy on test images: {100 * correct / total:.2f}%")